In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import types as T, functions as F

In [17]:
spark = SparkSession.builder.appName("practice").getOrCreate()
sc = spark.sparkContext

your 131072x1 screen size is bogus. expect trouble
25/03/20 14:00:29 WARN Utils: Your hostname, xRhl resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/20 14:00:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/20 14:00:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Sample DataFrame
df = spark.createDataFrame([("US",), ("IN",), ("CA",), ("FR",)], ["CountryCode"])
df.show()

+-----------+
|CountryCode|
+-----------+
|         US|
|         IN|
|         CA|
|         FR|
+-----------+



In [6]:
df.explain(True)

== Parsed Logical Plan ==
LogicalRDD [CountryCode#0], false

== Analyzed Logical Plan ==
CountryCode: string
LogicalRDD [CountryCode#0], false

== Optimized Logical Plan ==
LogicalRDD [CountryCode#0], false

== Physical Plan ==
*(1) Scan ExistingRDD[CountryCode#0]



In [4]:
df = df.repartition(1)
df.show()
df = df.repartition(4)
df.show()

+-----------+
|CountryCode|
+-----------+
|         US|
|         IN|
|         CA|
|         FR|
+-----------+

+-----------+
|CountryCode|
+-----------+
|         CA|
|         US|
|         IN|
|         FR|
+-----------+



In [5]:
def country_name(code):
    if code == "US":
        return "United States"
    elif code == "IN":
        return "India"
    elif code == "CA":
        return "Canada"
    else:
        return "Unknown"

country_name_udf = F.udf(country_name, T.StringType())
spark.udf.register("country_name_udf", country_name, T.StringType())

<function __main__.country_name(code)>

In [6]:
df.withColumn("country_name", F.expr("country_name_udf(CountryCode)")).show()


+-----------+-------------+
|CountryCode| country_name|
+-----------+-------------+
|         US|United States|
|         IN|        India|
|         CA|       Canada|
|         FR|      Unknown|
+-----------+-------------+



In [7]:
df.rdd.getNumPartitions()

4

In [13]:
spark.stop()

In [6]:
dir(df.rdd)

['__add__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getnewargs__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_computeFractionForSampleSize',
 '_defaultReducePartitions',
 '_id',
 '_is_barrier',
 '_is_protocol',
 '_jrdd',
 '_jrdd_deserializer',
 '_memory_limit',
 '_pickled',
 '_reserialize',
 '_to_java_object_rdd',
 'aggregate',
 'aggregateByKey',
 'barrier',
 'cache',
 'cartesian',
 'checkpoint',
 'cleanShuffleDependencies',
 'coalesce',
 'cogroup',
 'collect',
 'collectAsMap',
 'collectWithJobGroup',
 'combineByKey',
 'context',
 'count',
 'countApprox',
 'countApproxDistinct',
 'countByKey',
 'countByValue',
 'ctx',
 'dist

In [10]:
df.rdd

MapPartitionsRDD[24] at javaToPython at NativeMethodAccessorImpl.java:0

In [13]:
rdd = sc.parallelize([1, 2, 3, 4, 5])
rdd.collect()

[1, 2, 3, 4, 5]

In [15]:
rdd.coalesce(1)

CoalescedRDD[28] at coalesce at NativeMethodAccessorImpl.java:0

In [17]:
rdd = sc.parallelize([1, 2, 3, 4], 2)
rdd.getNumPartitions()

2

In [18]:
# Define a function to process each partition
def process_partition(iterator):
    print("Processing partition...")
    return [x * 2 for x in iterator]  # Multiply each element by 2

# Apply mapPartitions
result_rdd = rdd.mapPartitions(process_partition)

In [21]:
rdd.collect()

[1, 2, 3, 4]

In [20]:
result_rdd.collect()

Processing partition...
Processing partition...


[2, 4, 6, 8]

In [46]:
spark.sql(
    '''
    SELECT 
        struct(
        id idd,
        id idx
        ) stru,
    array(
        id, id, id
    ) arr,
    map(
        id, id,
        id+1, id+1
    ) mapp
    FROM RANGE(10)
'''
).show()

+------+---------+------------------+
|  stru|      arr|              mapp|
+------+---------+------------------+
|{0, 0}|[0, 0, 0]|  {0 -> 0, 1 -> 1}|
|{1, 1}|[1, 1, 1]|  {1 -> 1, 2 -> 2}|
|{2, 2}|[2, 2, 2]|  {2 -> 2, 3 -> 3}|
|{3, 3}|[3, 3, 3]|  {3 -> 3, 4 -> 4}|
|{4, 4}|[4, 4, 4]|  {4 -> 4, 5 -> 5}|
|{5, 5}|[5, 5, 5]|  {5 -> 5, 6 -> 6}|
|{6, 6}|[6, 6, 6]|  {6 -> 6, 7 -> 7}|
|{7, 7}|[7, 7, 7]|  {7 -> 7, 8 -> 8}|
|{8, 8}|[8, 8, 8]|  {8 -> 8, 9 -> 9}|
|{9, 9}|[9, 9, 9]|{9 -> 9, 10 -> 10}|
+------+---------+------------------+



In [62]:
spark.sql(
    '''
    SELECT 
        id,
        struct(
        id idd,
        id idx
        ) stru,
    array(
        id, id, id
    ) arr,
    map(
        id, id,
        id+1, id+1
    ) mapp
    FROM RANGE(10)
'''
).printSchema()

root
 |-- id: long (nullable = false)
 |-- stru: struct (nullable = false)
 |    |-- idd: long (nullable = false)
 |    |-- idx: long (nullable = false)
 |-- arr: array (nullable = false)
 |    |-- element: long (containsNull = false)
 |-- mapp: map (nullable = false)
 |    |-- key: long
 |    |-- value: long (valueContainsNull = false)



In [58]:
spark.sql(
    '''
    SELECT 
        id,
        struct(
        id idd,
        id idx
        ) stru,
    array(
        id, id, id
    ) arr,
    map(
        id, id,
        id+1, id+1
    ) mapp
    FROM RANGE(10)
'''
).repartition(1).write.format('json').mode('overwrite').save('/mnt/c/Users/raulr/OneDrive/Works/learning/de-skill/de-devspace/data-engineering/2025/core-concepts/data-processing/ddl/data/output/exjson')

In [60]:
djson = spark.sql(
    '''
    SELECT 
        id,
        struct(
        id idd,
        id idx
        ) stru,
    array(
        id, id, id
    ) arr,
    map(
        id, id,
        id+1, id+1
    ) mapp
    FROM RANGE(10)
'''
).toJSON()
import json
jd = json.loads(djson.first())
jd


{'id': 0,
 'stru': {'idd': 0, 'idx': 0},
 'arr': [0, 0, 0],
 'mapp': {'0': 0, '1': 1}}

In [29]:
df.createOrReplaceTempView("people")
df.show(truncate=False)

+-------+---+-----------+--------------+------------------------------------------------+
|name   |age|city       |skills        |contact                                         |
+-------+---+-----------+--------------+------------------------------------------------+
|Alice  |30 |New York   |[Python, SQL] |{phone -> 123-456, email -> alice@example.com}  |
|Bob    |25 |Los Angeles|[Java, Scala] |{phone -> 987-654, email -> bob@example.com}    |
|Charlie|35 |Chicago    |[Python, Java]|{phone -> 555-555, email -> charlie@example.com}|
+-------+---+-----------+--------------+------------------------------------------------+



In [32]:
dff = spark.sql('''
SELECT name, age, struct(city, age) AS address 
FROM people
''')

In [34]:
dir(dff.rdd)

['__add__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getnewargs__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_computeFractionForSampleSize',
 '_defaultReducePartitions',
 '_id',
 '_is_barrier',
 '_is_protocol',
 '_jrdd',
 '_jrdd_deserializer',
 '_memory_limit',
 '_pickled',
 '_reserialize',
 '_to_java_object_rdd',
 'aggregate',
 'aggregateByKey',
 'barrier',
 'cache',
 'cartesian',
 'checkpoint',
 'cleanShuffleDependencies',
 'coalesce',
 'cogroup',
 'collect',
 'collectAsMap',
 'collectWithJobGroup',
 'combineByKey',
 'context',
 'count',
 'countApprox',
 'countApproxDistinct',
 'countByKey',
 'countByValue',
 'ctx',
 'dist